<a href="https://colab.research.google.com/github/zune2/colab-notebooks/blob/main/2026-10-11-EP005-laya-style-test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import random
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel, AutoTokenizer

# ========================================================
# 1. 인프라 초기화 및 난수(Seed) 고정 (재현성 확보)
# ========================================================
def seed_everything(seed=2026):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(2026)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📡 [시스템 점검] 인프라 가속 디바이스: {device.type.upper()}")
if device.type == 'cpu':
    print("⚠️ 경고: 현재 CPU 가속 상태입니다. 속도 개선을 위해 캐글 우측 사이드바(Accelerator)에서 GPU 설정을 켜주세요!\n")


# ========================================================
# 2. PureLayaEngine 클래스 아키텍처 정의 (ModernBERT 인코더 기반)
# ========================================================
class PureLayaEngine(nn.Module):
    def __init__(self, backbone_model_name="answerdotai/ModernBERT-base", num_classes=4):
        super(PureLayaEngine, self).__init__()
        self.encoder = AutoModel.from_pretrained(backbone_model_name)
        self.hidden_size = self.encoder.config.hidden_size

        # Laya의 고유 명세인 Typed Decision Head 구현
        self.decision_head = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_size // 2, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_representation = outputs.last_hidden_state[:, 0, :]
        logits = self.decision_head(cls_representation)
        probabilities = torch.softmax(logits, dim=1)
        return logits, probabilities


# ========================================================
# 3. 모델 및 토크나이저 에셋 할당
# ========================================================
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
model = PureLayaEngine(num_classes=4).to(device)


# ========================================================
# 💡 4. [요청 사항 지점] 의사결정 레이어 교정을 위한 40개 정밀 지도 데이터셋 구축
# ========================================================
choices = ["Account/Auth", "Billing/Refund", "Technical Error", "UI/Feedback"]
label_map = {choices[i]: i for i in range(4)}

# 클래스별 10개씩 완벽한 문맥 패턴을 가진 총 40개의 하이엔드 영문 학습 샘플
train_samples = [
    # ---- Account/Auth (10 Samples) ----
    ("My login attempts keep failing and I am locked out of my profile.", "Account/Auth"),
    ("I cannot change my registered email address in the security tab.", "Account/Auth"),
    ("The two-factor authentication (2FA) verification code is not arriving.", "Account/Auth"),
    ("How do I link my business account with my corporate Google sign-on?", "Account/Auth"),
    ("I suspicious unauthorized access to my profile, reset all sessions.", "Account/Auth"),
    ("My session expires too quickly, forcing me to re-authenticate constantly.", "Account/Auth"),
    ("I forgot my security questions and cannot access my personal vault.", "Account/Auth"),
    ("Where can I update my account password and enable biometric keys?", "Account/Auth"),
    ("Can I change my account username or merge two different user profiles?", "Account/Auth"),
    ("My active user status shows deactivated but I never requested it.", "Account/Auth"),

    # ---- Billing/Refund (10 Samples) ----
    ("I was double charged on my visa credit card for this month's renewal.", "Billing/Refund"),
    ("I need to request a complete refund for the duplicate subscription fee.", "Billing/Refund"),
    ("Where can I download the formal tax invoice and receipt for my team purchase?", "Billing/Refund"),
    ("My corporate credit card payment keeps getting rejected at checkout.", "Billing/Refund"),
    ("Please cancel my recurring monthly premium plan immediately.", "Billing/Refund"),
    ("I want to update my billing cycle from monthly to an annual discount package.", "Billing/Refund"),
    ("My bank statement shows a charge from your service but my account is basic.", "Billing/Refund"),
    ("Do you support local wire transfers or corporate procurement cards?", "Billing/Refund"),
    ("I requested a chargeback but haven't received any confirmation email yet.", "Billing/Refund"),
    ("Why is there a hidden transaction fee applied to my international payment?", "Billing/Refund"),

    # ---- Technical Error (10 Samples) ----
    ("The backend server crashed with a fatal memory allocation segfault dump.", "Technical Error"),
    ("An unhandled exception occurs when executing the database synchronization query.", "Technical Error"),
    ("The image upload button triggers an immediate app crash on iOS devices.", "Technical Error"),
    ("API token verification keeps returning 502 Bad Gateway and 504 Timeouts.", "Technical Error"),
    ("The Docker container fails to bootstrap due to a missing network driver socket.", "Technical Error"),
    ("MacBook M4 build logs show a severe compilation error within your package.", "Technical Error"),
    ("Data export script is stuck at 99% and eventually throws an out-of-bounds error.", "Technical Error"),
    ("The websocket connection drops every 5 seconds with an unhandled protocol leak.", "Technical Error"),
    ("The system cache is corrupted, causing the dashboard to show blank raw code.", "Technical Error"),
    ("Our local staging environment cannot reach the authentication cluster port.", "Technical Error"),

    # ---- UI/Feedback (10 Samples) ----
    ("The contrast ratio of the current dark mode design is making it unreadable.", "UI/Feedback"),
    ("Please consider adding a custom drag-and-drop grid layout for dashboard panels.", "UI/Feedback"),
    ("The main navigation menu has too many nested sub-menus, simplify the layout.", "UI/Feedback"),
    ("The web interface looks completely unoptimized and broken on mobile Safari.", "UI/Feedback"),
    ("Font scale is too small on high-resolution 4K monitors, add a zooming feature.", "UI/Feedback"),
    ("The sidebar takes up too much horizontal space, please make it collapsible.", "UI/Feedback"),
    ("I highly appreciate the new analytical charts, they look stunning and helpful!", "UI/Feedback"),
    ("The animation transition between page loads is laggy and hurts UX fluidity.", "UI/Feedback"),
    ("Could you move the search input bar to the top-center position for clarity?", "UI/Feedback"),
    ("The buttons on the tablet viewport overlap with each other, blocking user clicks.", "UI/Feedback")
]


# ========================================================
# 5. 가중치 파인튜닝 (Fine-tuning) 가동 - Softmax 의사결정 수렴
# ========================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=4e-5, weight_decay=1e-2)

model.train()
print(f"⚙️ [학습 엔진 가동] 총 {len(train_samples)}개의 라벨링 문맥 기반 Decision Head 최적화 중...")

# 40개 대량 샘플의 정밀 수렴을 위해 6에폭 가동
for epoch in range(6):
    epoch_loss = 0.0
    # 데이터 순서를 섞어 과적합 방지
    random.shuffle(train_samples)

    for text, label_str in train_samples:
        optimizer.zero_grad()
        prompt = f"Context: {text} | Choices: {', '.join(choices)}"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)

        logits, _ = model(inputs['input_ids'], inputs['attention_mask'])
        loss = criterion(logits, torch.tensor([label_map[label_str]]).to(device))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch + 1) % 2 == 0:
        print(f"   ▶ Epoch {epoch+1}/6 | Total Loss Value: {epoch_loss:.4f}")

print("✅ 튜닝 완전 완료! 이제 의사결정 엔진이 문맥적 차원 분리 능력을 획득했습니다.\n")


📡 [시스템 점검] 인프라 가속 디바이스: CUDA


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚙️ [학습 엔진 가동] 총 40개의 라벨링 문맥 기반 Decision Head 최적화 중...
   ▶ Epoch 2/6 | Total Loss Value: 56.3298
   ▶ Epoch 4/6 | Total Loss Value: 50.6875
   ▶ Epoch 6/6 | Total Loss Value: 8.3973
✅ 튜닝 완전 완료! 이제 의사결정 엔진이 문맥적 차원 분리 능력을 획득했습니다.



In [ ]:
# 4. 20개의 라이브 평가 데이터셋 및 실제 정답(Ground Truth) 동기화 결합
test_traffic_dataset = [
    ("I am trying to log in but the password reset email never arrived. My spam folder is empty.", "Account/Auth"),
    ("I accidentally double-clicked the subscription button. Can you cancel one of the charges immediately?", "Billing/Refund"),
    ("A dependency library conflict error occurs when running the server on a MacBook M3. Attaching the logs.", "Technical Error"),
    ("The readability of the dashboard dark mode is a bit poor. Please make the font color slightly brighter.", "UI/Feedback"),
    ("I want to disconnect the Google social login link associated with my existing account.", "Account/Auth"),
    ("Does the premium plan automatically charge after the one-month free trial ends?", "Billing/Refund"),
    ("The loading spinner keeps spinning on the payment completion screen and it won't redirect. Money was deducted.", "Technical Error"),
    ("The mobile app crashes unexpectedly whenever I try to press the image upload button.", "Technical Error"),
    ("When using the data export feature, Korean fonts in the Excel file appear corrupted and unreadable.", "Technical Error"),
    ("I am wondering if additional discount benefits or a separate tax invoice can be issued for corporate bulk purchases.", "Billing/Refund"),
    ("It would be great to have a feature where users can freely rearrange the positions of the main dashboard widgets.", "UI/Feedback"),
    ("Is there a way to backup and transfer data from my old device directly to my new smartphone?", "Account/Auth"),
    ("It has been 3 days since I requested a refund, but the money has not hit my bank account yet. Please check.", "Billing/Refund"),
    ("The API call token authentication keeps throwing a '401 Unauthorized' error. Is the server undergoing maintenance?", "Technical Error"),
    ("The main menu tab structure is too complicated and hard to read. Please simplify the UI design.", "UI/Feedback"),
    ("I want to delete my account, but I cannot find a delete or deactivate button anywhere in my profile page.", "Account/Auth"),
    ("Can I pay with a local debit card? I don't have an international credit card right now.", "Billing/Refund"),
    ("I turned off all push notifications in the settings, but real-time marketing text messages keep coming in.", "UI/Feedback"),
    ("The site layout is completely broken on my tablet PC screen, making it impossible to click any buttons.", "UI/Feedback"),
    ("The service is amazing! Especially the AI prediction performance updated this time has become incredibly good.", "UI/Feedback")
]

# 5. 실시간 루프(for loop) 구동 및 정량 지표 계측
model.eval()
print(f"📦 [실시간 프로덕션 테스트] 총 {len(test_traffic_dataset)}개 페이로드 검증 시작...")
print("-" * 110)

correct_predictions = 0
total_start_time = time.time()

for idx, (query, ground_truth) in enumerate(test_traffic_dataset, start=1):
    loop_start_time = time.time()

    prompt = f"Context: {query} | Question: Which category does this belong to? | Choices: {', '.join(choices)}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)

    with torch.no_grad():
        _, probs = model(inputs['input_ids'], inputs['attention_mask'])
        # 💡 [교정 지점]: 뒤에 [0]을 붙여 2차원 배치를 깔끔한 1차원 넘파이 배열로 변환합니다.
        probs_numpy = probs.cpu().numpy()[0]

    best_choice_idx = np.argmax(probs_numpy)
    predicted_label = choices[best_choice_idx]
    predicted_confidence = probs_numpy[best_choice_idx] # 이제 에러 없이 단일 확률값을 정상 호출합니다.

    is_correct = (predicted_label == ground_truth)
    if is_correct:
        correct_predictions += 1
        match_symbol = "🟩 [정답]"
    else:
        match_symbol = "🟥 [오답]"

    loop_end_time = time.time()
    latency_ms = (loop_end_time - loop_start_time) * 1000

    print(f"[{idx:02d}] {match_symbol} Query: '{query[:55]}...'")
    print(f"    ↳ 🎯 실제정답: {ground_truth:<15} | 🤖 AI예측: {predicted_label:<15} (Conf: {predicted_confidence * 100:.2f}%) | Latency: {latency_ms:.2f}ms")
    print("-" * 110)

total_duration = time.time() - total_start_time
average_latency = (total_duration / len(test_traffic_dataset)) * 1000
final_accuracy = (correct_predictions / len(test_traffic_dataset)) * 100

print("\n" + "="*80)
print("🏆 [인프라 및 모델 예측 검증 종합 모니터링 리포트]")
print(f"📊 최종 분류 정확도 (Final Accuracy) : {final_accuracy:.2f} % ({correct_predictions}/{len(test_traffic_dataset)} 개 적중)")
print(f"⏱️ 평균 추론 속도 (Mean Latency)       : {average_latency:.2f} ms")
print(f"⏳ 총 가동 연산 시간 (Total Duration)  : {total_duration:.4f} 초")
print("="*80)


📦 [실시간 프로덕션 테스트] 총 20개 페이로드 검증 시작...
--------------------------------------------------------------------------------------------------------------
[01] 🟩 [정답] Query: 'I am trying to log in but the password reset email neve...'
    ↳ 🎯 실제정답: Account/Auth    | 🤖 AI예측: Account/Auth    (Conf: 85.74%) | Latency: 33.21ms
--------------------------------------------------------------------------------------------------------------
[02] 🟩 [정답] Query: 'I accidentally double-clicked the subscription button. ...'
    ↳ 🎯 실제정답: Billing/Refund  | 🤖 AI예측: Billing/Refund  (Conf: 61.39%) | Latency: 29.87ms
--------------------------------------------------------------------------------------------------------------
[03] 🟩 [정답] Query: 'A dependency library conflict error occurs when running...'
    ↳ 🎯 실제정답: Technical Error | 🤖 AI예측: Technical Error (Conf: 97.44%) | Latency: 28.38ms
--------------------------------------------------------------------------------------------------------------
[04] 🟩 [정